# BioGPT (original)

Baseline from the original paper — 24 layers, 1024-dim.

**Model type:** Decoder (causal LM)  
**Pipeline:** Probing → Orthogonalization → Steering sweep  
**Results saved to:** `results/biogpt/`

## 1. Setup

In [ ]:
import sys, json
from pathlib import Path

# ── ensure project root is on the path ──────────────────────────────────────
ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.model_registry  import get_config, list_models
from src.model_loader    import load_model, get_device
from src.bioscope_parser import build_balanced_contrast_set
from src.probing         import run_probing_sweep, best_layer
from src.orthogonalization import build_orthogonal_directions
from src.steering_core   import calibrate_hidden_norms
from src.experiment_runner import run_experiment, PROMPTS, ALPHAS, SEEDS
from src.results_manager import save_results, load_results, results_exist
from src.metrics         import load_english_vocab

print(f"PyTorch   : {torch.__version__}")
print(f"Device    : {get_device()}")


In [ ]:
MODEL_NAME   = "biogpt"
DISPLAY_NAME = "BioGPT (original)"
cfg          = get_config(MODEL_NAME)
DEVICE       = get_device()
print(cfg)

## 2. Load model

In [ ]:
tok, model = load_model(MODEL_NAME, device=DEVICE)
print(f"Loaded: {cfg['hf_id']}")

## 3. Load BioScope data

In [ ]:
# ── BioScope contrast set ────────────────────────────────────────────────────
# seed=42 is fixed from the original paper — do not change
uncertain, certain = build_balanced_contrast_set(
    "data/bioscope", max_per_class=200, seed=42
)
print(f"Loaded {len(uncertain)} uncertain + {len(certain)} certain sentences")
print("\nUncertain sample:")
for s in uncertain[:2]:
    print(f"  {s[:100]}")
print("\nCertain sample:")
for s in certain[:2]:
    print(f"  {s[:100]}")


## 4. Probing sweep — all layers

Train a logistic regression at each layer and measure 5-fold CV accuracy.

In [ ]:
from src.probing import run_probing_sweep, best_layer

probe_results = run_probing_sweep(
    uncertain, certain, tok, model,
    layers=cfg["probe_layers"], device=DEVICE,
)

### Best layer

In [ ]:
selected_layer = cfg["steer_layer"] or best_layer(probe_results)
print(f"Selected layer: {selected_layer}  "  
      f"acc={probe_results[selected_layer]['mean_acc']:.2%}")

## 5. Orthogonalization

Remove length and lexical-hedge confounds from the probe direction.

In [ ]:
from src.orthogonalization import build_orthogonal_directions

ortho = build_orthogonal_directions(
    uncertain, certain, tok, model, selected_layer,
    device=DEVICE,
)
print(f"  cos(probe, length) = {ortho['cos_probe_length']:.4f}")
print(f"  cos(probe, hedge)  = {ortho['cos_probe_hedge']:.4f}")
print(f"  ortho_LH acc       = {ortho['ortho_LH_classification_acc']:.2%}")

## 6. Calibrate hidden norms

In [ ]:
hidden_norms = calibrate_hidden_norms(
    tok, model, MODEL_NAME, layers=[selected_layer], device=DEVICE,
)
print(f"Hidden norm at layer {selected_layer}: {hidden_norms[selected_layer]:.1f}")

## 7. Full steering experiment

Sweeps 9 α values × 20 seeds × 5 prompts × 2 directions = 1 800 generations.

> **Note:** This cell is compute-intensive. Skip and load from disk if already run (next cell checks).

In [ ]:
if results_exist(MODEL_NAME, "experiment"):
    print("Loading cached results...")
    results = load_results(MODEL_NAME, "experiment")
else:
    results = run_experiment(
        MODEL_NAME, tok, model, uncertain, certain,
        device=DEVICE,
        steer_layer=selected_layer,
    )
    save_results(MODEL_NAME, "experiment", results)
print(f"Records: {len(results['records'])}")

## 8. Visualizations

In [ ]:
# ── probe accuracy plot ───────────────────────────────────────────────────────
probe_res = results["probe_results"]
layers_sorted = sorted(int(k) for k in probe_res.keys())
means  = [probe_res[str(l)]["mean_acc"] for l in layers_sorted]
cis_lo = [probe_res[str(l)]["ci_low"]   for l in layers_sorted]
cis_hi = [probe_res[str(l)]["ci_high"]  for l in layers_sorted]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(layers_sorted, means, marker="o", linewidth=2, label="CV accuracy")
ax.fill_between(layers_sorted, cis_lo, cis_hi, alpha=0.2, label="95% CI")
ax.axhline(0.5, color="grey", linestyle="--", linewidth=1, label="Chance")
ax.set_xlabel("Layer")
ax.set_ylabel("Accuracy")
ax.set_title(f"Probe accuracy per layer — {DISPLAY_NAME}")
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()

fig_path = Path("results") / MODEL_NAME / "fig_probe_accuracy.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
fig.savefig(fig_path.with_suffix(".pdf"), bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")


In [ ]:
# ── steering trade-off plot (decoder) ────────────────────────────────────────
summary = results["summary"]
directions = list(summary.keys())
metrics_to_plot = ["hedge_score", "perplexity", "lexical_diversity", "token_validity"]
metric_labels   = ["Hedge score", "Perplexity", "Lexical diversity", "Token validity"]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(14, 4))
colors = {"probe": "#2196F3", "ortho": "#FF5722"}
markers = {"probe": "o", "ortho": "s"}

for ax, metric, label in zip(axes, metrics_to_plot, metric_labels):
    for dir_name in directions:
        dir_data = summary[dir_name]
        alphas_sorted = sorted(float(a) for a in dir_data.keys())
        y_means = [dir_data[str(a)].get(f"{metric}_mean", np.nan) for a in alphas_sorted]
        y_stds  = [dir_data[str(a)].get(f"{metric}_std",  0.0)    for a in alphas_sorted]
        ax.plot(alphas_sorted, y_means,
                marker=markers[dir_name], color=colors[dir_name],
                label=dir_name, linewidth=2)
        ax.fill_between(
            alphas_sorted,
            [m - s for m, s in zip(y_means, y_stds)],
            [m + s for m, s in zip(y_means, y_stds)],
            alpha=0.15, color=colors[dir_name],
        )
    ax.set_xlabel("α fraction")
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle(f"Steering trade-offs — {DISPLAY_NAME}", fontsize=12)
plt.tight_layout()

fig_path = Path("results") / MODEL_NAME / "fig_steering_tradeoff.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
fig.savefig(fig_path.with_suffix(".pdf"), bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")


## 9. Sample steered outputs

In [ ]:
import random
recs = [r for r in results["records"] if r["alpha_frac"] == 0.15
        and r["direction"] == "probe"]
for r in random.sample(recs, min(3, len(recs))):
    print(f"[α=0.15 probe | seed={r['seed']}]")
    print(f"  {r['generation'][:200]}")
    print(f"  hedge={r['hedge_score']:.2f}  ppl={r.get('perplexity','N/A')}")
    print()

## 10. Summary statistics

In [ ]:
summary = results["summary"]
header = "{:>8} {:>14} {:>14} {:>12} {:>12}".format("", "probe hedge", "ortho hedge", "probe ppl", "ortho ppl")
print(header)
for af in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25]:
    ph = summary.get("probe", {}).get(str(af), {}).get("hedge_score_mean", float("nan"))
    oh = summary.get("ortho", {}).get(str(af), {}).get("hedge_score_mean", float("nan"))
    pp = summary.get("probe", {}).get(str(af), {}).get("perplexity_mean", float("nan"))
    op = summary.get("ortho", {}).get(str(af), {}).get("perplexity_mean", float("nan"))
    print(f"  alpha={af:.3f}  {ph:>14.3f} {oh:>14.3f} {pp:>12.1f} {op:>12.1f}")
